# Results paragraph: Developmental (age) effects on microstructural measures

Uses assembled **age–quality effects** (no_quality only): ΔR²adj = full GAMM (age, sex, site) minus reduced model excluding age. Data: `data/age_quality_effects/age_quality_effects_all_outputs.rds`. Filter: `qc_metric == "no_quality"`, harmonized, non_vendorwise, five metrics (ICVF, MKT, RTOP, FA, MD).

**Fig. 4:** Panel A = age-effect heatmap (ΔR²adj); Panel B = ICVF; Panel C = MKT/RTOP. **Supplementary Fig. S4** = all measures.

## Setup and load age effects

In [ ]:
suppressPackageStartupMessages({
  library(dplyr)
  library(fs)
  library(jsonlite)
})

config_candidates <- c(
  Sys.getenv("CONFIG_PATH", unset = ""),
  fs::path(".", "config.json"),
  fs::path("..", "config.json"),
  fs::path("..", "..", "config.json")
)
config_candidates <- normalizePath(unique(config_candidates[nzchar(config_candidates)]), winslash = "/", mustWork = FALSE)
config_path <- config_candidates[file.exists(config_candidates)][1]
if (is.na(config_path) || !nzchar(config_path)) stop("Could not locate config.json.")
config <- jsonlite::fromJSON(config_path)
project_root <- normalizePath(config$project_root, winslash = "/", mustWork = FALSE)

age_rds <- fs::path(project_root, "data", "age_quality_effects", "age_quality_effects_all_outputs.rds")
if (!file.exists(age_rds)) stop("Age effects file not found: ", age_rds)

df_all <- readRDS(age_rds)
if (!is.data.frame(df_all)) stop("Age effects file is not a data.frame.")

qc_target <- "no_quality"
source_target <- "harmonized"
output_target <- "non_vendorwise_pairwise"
metrics_keep <- c("DKI_mkt", "NODDI_icvf", "MAPMRI_rtop", "GQI_fa", "GQI_md")
metric_labels <- c(
  DKI_mkt = "MKT",
  NODDI_icvf = "ICVF",
  MAPMRI_rtop = "RTOP",
  GQI_fa = "FA",
  GQI_md = "MD"
)

qc_col <- if ("qc_metric" %in% names(df_all)) "qc_metric" else "qc_covariate"
if (!qc_col %in% names(df_all) || !qc_target %in% df_all[[qc_col]]) stop("No no_quality rows in age effects.")
df <- df_all %>%
  filter(
    .data[[qc_col]] == qc_target,
    .data[["source"]] == source_target,
    .data[["output_type"]] == output_target,
    metric %in% metrics_keep,
    !is.na(.data[["age_effect_size"]])
  )

cat("N rows (no_quality, harmonized, non_vendorwise, five metrics):", nrow(df), "\n")

## Age effect (ΔR²adj) summary by metric

Min, max, and mean of age effect size (as % variance explained) across bundles, for paragraph placeholders.

In [ ]:
# age_effect_size is ΔR²adj (proportion); convert to % for paragraph
effect_summary <- df %>%
  mutate(effect_pct = .data[["age_effect_size"]] * 100) %>%
  group_by(metric) %>%
  summarise(
    min_pct = min(effect_pct, na.rm = TRUE),
    max_pct = max(effect_pct, na.rm = TRUE),
    mean_pct = mean(effect_pct, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  mutate(metric_label = unname(metric_labels[metric])) %>%
  arrange(desc(mean_pct))

cat("Age effect (ΔR²adj as % variance) across bundles by metric:\n")
print(effect_summary)

# One decimal for range/mean in text; format range as X.X–X.X or X–X
fmt_range <- function(mn, mx) {
  if (mn >= 10 || mx >= 10) sprintf("%.0f–%.0f", mn, mx) else sprintf("%.1f–%.1f", mn, mx)
}
fmt_mean <- function(x) sprintf("%.1f", x)

get_row <- function(met) {
  effect_summary %>% filter(metric == met) %>% slice(1)
}

icvf  <- get_row("NODDI_icvf")
mkt   <- get_row("DKI_mkt")
rtop  <- get_row("MAPMRI_rtop")
md    <- get_row("GQI_md")
fa    <- get_row("GQI_fa")

icvf_range  <- fmt_range(icvf$min_pct,  icvf$max_pct)
icvf_mean   <- fmt_mean(icvf$mean_pct)
mkt_range   <- fmt_range(mkt$min_pct,   mkt$max_pct)
mkt_mean    <- fmt_mean(mkt$mean_pct)
rtop_range  <- fmt_range(rtop$min_pct,  rtop$max_pct)
rtop_mean   <- fmt_mean(rtop$mean_pct)
md_range    <- fmt_range(md$min_pct,   md$max_pct)
md_mean     <- fmt_mean(md$mean_pct)
fa_range    <- fmt_range(fa$min_pct,   fa$max_pct)
fa_mean     <- fmt_mean(fa$mean_pct)

cat("\nParagraph placeholders:\n")
cat("  ICVF:  range", icvf_range, "%, mean", icvf_mean, "%\n")
cat("  MKT:   range", mkt_range, "%, mean", mkt_mean, "%\n")
cat("  RTOP:  range", rtop_range, "%, mean", rtop_mean, "%\n")
cat("  MD:    range", md_range, "%, mean", md_mean, "%\n")
cat("  FA:    range", fa_range, "%, mean", fa_mean, "%\n")

## Filled paragraph

In [ ]:
txt <- paste(
  "Multi-shell dMRI enables the estimation of numerous microstructural indices, yet most developmental studies have historically focused on only one measure, or limited subset. To systematically evaluate their relative sensitivity to development, we quantified the association between age and harmonized microstructural measures across white matter bundles. For each metric within each bundle, we fit generalized additive mixed models including age, sex, and site, and operationalized the developmental effect as ΔR²adj between the full model and a reduced model excluding age (Fig. 4a). Across the microstructural metrics tested, the strongest developmental effects were observed for the ICVF (Fig. 4b), followed by MKT and RTOP (Fig. 4c). ICVF age effects explained between", icvf_range, "% of variance across bundles (mean ΔR²adj =", icvf_mean, "%). MKT explained", mkt_range, "% (mean =", mkt_mean, "%), and RTOP explained", rtop_range, "% (mean =", rtop_mean, "%). In contrast, tensor-derived measures demonstrated substantially weaker developmental sensitivity. MD explained", md_range, "% (mean =", md_mean, "%), while fractional FA explained", fa_range, "% of variance across bundles (mean =", fa_mean, "%). Results for all microstructural measures are provided in Supplementary Fig. S4. Together, these results emphasize that there is pronounced heterogeneity in developmental sensitivity of features from diffusion models and indicate that advanced multi-compartment and non-Gaussian models capture markedly stronger age-related effects than conventional tensor-derived metrics.",
  sep = " "
)
cat(txt, "\n")